Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [1]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing some common hand gestures in the [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p) dataset.

## Prerequisites

Install the MediaPipe Model Maker package.

In [2]:
# !pip install --upgrade pip
!pip install mediapipe-model-maker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 KB 1.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 KB 5.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 KB 3.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 MB 4.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 5.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 5.0 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.1 MB/s eta 0:00:0000:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 2.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 KB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 3.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━

Import the required libraries.

In [3]:
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

2026-03-31 13:46:58.874349: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-31 13:46:58.913069: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-31 13:46:58.913112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-31 13:46:58.914633: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-31 13:46:58.921627: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-31 13:46:58.921906: I tensorflow/core/platform/cpu_feature_guard.cc:1

## Simple End-to-End Example

This end-to-end example uses Model Maker to customize a model for on-device gesture recognition.

### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This example uses a rock paper scissors dataset sample which is downloaded from GCS.

In [4]:
# !wget https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip
# !unzip rps_data_sample.zip
dataset_path = "/home/dzmitry/gesture_real_time_control/raw_hands_united_crop" 
export_dir = "models/uv1"

In [5]:
os.path.exists(dataset_path)


True

Verify the rock paper scissors dataset by printing the labels. There should be 4 gesture labels, with one of them being the `none` gesture.

In [6]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

/home/dzmitry/gesture_real_time_control/raw_hands_united_crop
['palm', 'none', 'rock']


To better understand the dataset, plot a couple of example images for each gesture.

### Run the example
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the dataset: 80% for training, 10% for validation, and 10% for testing.

In [7]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/palm/PXL_20260322_215802094.jpg
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-30 19-00-32.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-30 19-00-15.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-23 20-06-32_40002542.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/palm/2026-03-23 19-22-59_19850914.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-23 20-19-28_02915291.JPG


I0000 00:00:1774954026.168989   18936 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774954026.246630   20562 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.288.01), renderer: NVIDIA GeForce GTX 1070/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774954026.260839   20565 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774954026.280647   20564 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/dzmitry/gesture_real_time_control/.venv/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetProtot

INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/none/1128.jpg
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-27 17-35-30.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/palm/PXL_20260322_215053375.jpg
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-23 19-23-43.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-23 19-21-05_1774283880_86674376.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/palm/PXL_20260322_215746824.jpg
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/rock/2026-03-30 18-58-56_28307022.JPG
INFO:tensorflow:Loading image /home/dzmitry/gesture_real_time_control/raw_hands_united_crop/palm/2026-03-23 20-18-46.JPG
INFO:tensorf

2026-03-31 13:48:11.811619: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-31 13:48:11.812574: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:Load valid hands with size: 1221, num_label: 3, labels: none,palm,rock.


INFO:tensorflow:Load valid hands with size: 1221, num_label: 3, labels: none,palm,rock.


**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section below.

In [8]:
hparams = gesture_recognizer.HParams(export_dir=export_dir)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)




Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 3)                 387       
 out (Dense)                                                     
                                                             

INFO:tensorflow:Training the models...


Epoch 1/10
488/488 [==============================] - 2s 3ms/step - loss: 0.3508 - categorical_accuracy: 0.6014 - val_loss: 0.3938 - val_categorical_accuracy: 0.6885 - lr: 0.0010
Epoch 2/10
488/488 [==============================] - 1s 2ms/step - loss: 0.2796 - categorical_accuracy: 0.6598 - val_loss: 0.3139 - val_categorical_accuracy: 0.7623 - lr: 9.9000e-04
Epoch 3/10
488/488 [==============================] - 1s 2ms/step - loss: 0.2577 - categorical_accuracy: 0.6670 - val_loss: 0.2687 - val_categorical_accuracy: 0.7623 - lr: 9.8010e-04
Epoch 4/10
488/488 [==============================] - 1s 2ms/step - loss: 0.2413 - categorical_accuracy: 0.6865 - val_loss: 0.2441 - val_categorical_accuracy: 0.7869 - lr: 9.7030e-04
Epoch 5/10
488/488 [==============================] - 1s 2ms/step - loss: 0.2318 - categorical_accuracy: 0.7018 - val_loss: 0.2368 - val_categorical_accuracy: 0.7869 - lr: 9.6060e-04
Epoch 6/10
488/488 [==============================] - 1s 2ms/step - loss: 0.2249 - catego

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [9]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

123/123 [==============================] - 0s 839us/step - loss: 0.1557 - categorical_accuracy: 0.8293
Test loss:0.15573562681674957, Test accuracy:0.8292682766914368


**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [10]:
model.export_model()
# !ls exported_model

Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite
INFO:tensorflow:Assets written to: /tmp/tmpf3lmu389/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmpf3lmu389/saved_model/assets
2026-03-31 13:48:27.441241: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-31 13:48:27.441274: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-31 13:48:27.441553: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpf3lmu389/saved_model
2026-03-31 13:48:27.442582: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-31 13:48:27.442598: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpf3lmu389/saved_model
2026-03-31 13:48:27.444460: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-03-31 13:48:27.444954: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-31 13:48:27.461346: I tensorflow/cc/saved_model/